# HOPG slab preparation

Build a repeated slab from an input POSCAR, optionally remove the top layer, then export POSCAR and LAMMPS data files.

This notebook is no longer tied to an absolute local path. Configure the input and output paths below.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "python" / "lammps_md_preparation" / "__init__.py").exists():
    NOTEBOOK_DIR = cwd
    package_root = cwd / "python"
elif (cwd / "lammps_md_preparation" / "python" / "lammps_md_preparation" / "__init__.py").exists():
    NOTEBOOK_DIR = cwd / "lammps_md_preparation"
    package_root = NOTEBOOK_DIR / "python"
else:
    raise RuntimeError("Run this notebook from the repository root or from lammps_md_preparation/.")

if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))

from ase.io import write
from lammps_md_preparation.utils import build_slab_supercell, rewrite_lammps_atom_types, write_lammps_data_file


In [ ]:
# Configuration
INPUT_POSCAR = NOTEBOOK_DIR / "input" / "POSCAR_Unit_Cell"
OUTPUT_DIR = NOTEBOOK_DIR / "output" / "hopg"
REPEATS = (8, 8, 2)
VACUUM = 20.0
REMOVE_TOP_LAYER = True
LAYER_TOLERANCE = 0.5  # Z-spacing threshold used to identify the top graphene layer.
SORT_BY_Z = True

# Optional HOPG-specific override for downstream LAMMPS conventions.
FORCE_ALL_ATOM_TYPES_TO = None  # Set an integer to force one atom type for every atom.
DECLARE_ATOM_TYPE_COUNT = None  # Override the declared number of atom types in the header.

BASENAME = f"HOPG_{REPEATS[0]}x{REPEATS[1]}x{REPEATS[2]}"


In [ ]:
supercell = build_slab_supercell(
    input_poscar=INPUT_POSCAR,
    repeats=REPEATS,
    vacuum=VACUUM,
    remove_top_layer=REMOVE_TOP_LAYER,
    layer_tolerance=LAYER_TOLERANCE,
    sort_by_z=SORT_BY_Z,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
poscar_path = OUTPUT_DIR / f"POSCAR_{BASENAME}"
raw_lammps_path = OUTPUT_DIR / f"lmp_{BASENAME}.lmp"
write(poscar_path, supercell, format="vasp", vasp5=True, direct=True)
write_lammps_data_file(supercell, raw_lammps_path)

final_lammps_path = raw_lammps_path
if FORCE_ALL_ATOM_TYPES_TO is not None or DECLARE_ATOM_TYPE_COUNT is not None:
    final_lammps_path = OUTPUT_DIR / f"lmp_{BASENAME}_rewritten.lmp"
    rewrite_lammps_atom_types(
        raw_lammps_path,
        final_lammps_path,
        atom_type_count=DECLARE_ATOM_TYPE_COUNT,
        atom_type_value=FORCE_ALL_ATOM_TYPES_TO,
    )

print(f"Generated POSCAR: {poscar_path}")
print(f"Generated LAMMPS data: {final_lammps_path}")
print(f"Atom count: {len(supercell)}")
